Drop the redundant columns like: seller_id — street — postal_code.

Drop the columns with > 60% missing values like (balcony, swimming pool, garden, energy_consumption, garage and terrace)

------------

In [1]:
import pandas as pd

sale_df = pd.read_csv("../data/SaleCleanForAnalysis.csv")

In [2]:
#1. Check and remove duplicate rows (new — not covered yet)

# How many exact duplicate rows exist?
print("Duplicate rows:", sale_df.duplicated().sum())

# Inspect a few to confirm they're true duplicates
print(sale_df[sale_df.duplicated(keep=False)].sort_values(by=sale_df.columns[0]).head(10))

# Drop them
sale_df = sale_df.drop_duplicates()
print("Shape after dropping duplicates:", sale_df.shape)

Duplicate rows: 0
Empty DataFrame
Columns: [longitude, latitude, transaction_type, price, property_type, property_subtype, seller_id, postal_code, date_of_construction, property_condition, livable_surface, number_of_bedrooms, number_of_bathrooms, elevator, terrace, furnished, availability, province, street, street_number, garage, land_surface, energy_consumption, garden, balcony, swimming_pool]
Index: []

[0 rows x 26 columns]
Shape after dropping duplicates: (9705, 26)


In [3]:
# e.g. same address + price + surface = likely the same listing re-scraped
subset_cols = ["latitude", "longitude", "price", "livable_surface"]
print("Likely duplicate listings:", sale_df.duplicated(subset=subset_cols).sum())

Likely duplicate listings: 135


In [4]:
# 2. Drop the columns you've already identified

drop_cols = ["seller_id", "street", "postal_code", "transaction_type", "garden", "energy_consumption", "street_number",
             "garage", "terrace"]

sale_df_clean = sale_df.drop(columns=drop_cols)
print(sale_df_clean.shape)

(9705, 17)


In [5]:
# 3 true boolean flags - missing means "not present"
	
binary_cols = ["balcony", "swimming_pool","elevator","furnished"]
sale_df_clean[binary_cols] = sale_df_clean[binary_cols].fillna(False)
# confirm
print(sale_df_clean.isnull().sum())

longitude                  3
latitude                   3
price                      0
property_type              0
property_subtype           0
date_of_construction    3425
property_condition      2195
livable_surface          553
number_of_bedrooms       174
number_of_bathrooms      930
elevator                   0
furnished                  0
availability            4381
province                   0
land_surface            4070
balcony                    0
swimming_pool              0
dtype: int64


In [6]:
# 4. Clean up availability (from the last check)
def simplify_availability(val):
    if pd.isna(val):
        return "Unknown"
    if val in ["On contract", "Immediately", "Negotiable"]:
        return val
    if val.startswith("From"):
        return "Future date"
    return val

sale_df_clean["availability_clean"] = sale_df_clean["availability"].apply(simplify_availability)
sale_df_clean = sale_df_clean.drop(columns=["availability"])  # optional, drop the raw messy column
print(sale_df_clean["availability_clean"].value_counts())


availability_clean
Unknown        4381
On contract    4061
Immediately     864
Negotiable      305
Future date      94
Name: count, dtype: int64


In [7]:
# 5. Drop the small number of rows missing lat/long (only 3 originally)
subset_cols = ["latitude", "longitude", "price", "livable_surface"]
sale_df_clean = sale_df_clean.dropna(subset=subset_cols)

In [8]:
print("Before final dedup:", sale_df_clean.shape)
sale_df_clean = sale_df_clean.drop_duplicates()
print("After final dedup:", sale_df_clean.shape)

Before final dedup: (9149, 17)
After final dedup: (9106, 17)


In [9]:
# 6. Final verification — should show all zeros

print(sale_df_clean.isna().sum())
print(sale_df_clean.shape)
print("Remaining duplicates:", sale_df_clean.duplicated().sum())

longitude                  0
latitude                   0
price                      0
property_type              0
property_subtype           0
date_of_construction    3072
property_condition      1948
livable_surface            0
number_of_bedrooms       134
number_of_bathrooms      734
elevator                   0
furnished                  0
province                   0
land_surface            3809
balcony                    0
swimming_pool              0
availability_clean         0
dtype: int64
(9106, 17)
Remaining duplicates: 0


In [10]:
sale_df_clean["land_surface"].median()
print("Median of land_surface:", sale_df_clean["land_surface"].median())
print("Missing land_surface:", sale_df_clean["land_surface"].isna().sum())
print("Total rows:", len(sale_df_clean))

Median of land_surface: 573.0
Missing land_surface: 3809
Total rows: 9106


# Handeling missing data!

In [11]:
# 7. Impute remaining missing values
numeric_cols = ["livable_surface", "number_of_bedrooms", "number_of_bathrooms",
                 "land_surface", "date_of_construction"]

categorical_cols = ["property_type", "property_subtype", "property_condition",
                     "province", "availability_clean", "elevator", "furnished"]

# Numeric → median impute
for col in numeric_cols:
    sale_df_clean[col] = sale_df_clean[col].fillna(sale_df_clean[col].median())

# Categorical → placeholder
for col in categorical_cols:
    sale_df_clean[col] = sale_df_clean[col].fillna("Unknown")

In [12]:
#7. Save and commit
sale_df_clean.to_csv("../data/SaleCleanFinal.csv", index=False)

----------

## Encoding and scaling!

Target:

price

Categorical (one-hot encode):

property_type, property_subtype, property_condition, province, availability_clean

Binary flags (convert to 0/1, no scaling):

elevator, furnished, balcony, swimming_pool

Numeric (scale):

livable_surface, number_of_bedrooms, number_of_bathrooms, land_surface, date_of_construction, latitude, longitude

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#3. split features/Target
X = sale_df_clean.drop(columns=["price"])
y = sale_df_clean["price"]

# 4. Train/test split Before scaling-avoid leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 1. convert boolean flags to clean 0/1
binary_cols = ["elevator", "furnished", "balcony", "swimming_pool"]
sale_df_clean[binary_cols]= sale_df_clean[binary_cols].astype(int)

#2. One-hot encode categoricals
categorical_cols = ["property_type", "property_subtype", "property_condition",
                     "province", "availability_clean"]

# Fit the encoder on TRAIN categorical columns only
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded_array_train = encoder.fit_transform(X_train[categorical_cols])
encoded_array_test = encoder.transform(X_test[categorical_cols])          # <-- ADDED: encode test too

# Turn the arrays into labeled DataFrames, aligned to the correct index
features_train = pd.DataFrame(encoded_array_train, columns=encoder.get_feature_names_out(categorical_cols),
                               index=X_train.index)                       # <-- FIXED: X_train.index, not sale_df_clean.index
features_test = pd.DataFrame(encoded_array_test, columns=encoder.get_feature_names_out(categorical_cols),
                              index=X_test.index)                         # <-- ADDED

# Drop the raw string columns and attach the one-hot columns
X_train = pd.concat([X_train.drop(columns=categorical_cols), features_train], axis=1)   # <-- ADDED
X_test = pd.concat([X_test.drop(columns=categorical_cols), features_test], axis=1)      # <-- ADDED

# Simulate new, unseen data arriving — the encoder handles it consistently
new_listings = pd.DataFrame({
    "property_type": ["Appartment", "House"],
    "property_subtype": ["Flat", "Villa"],
    "property_condition": ["Excellent", "To renovate"],
    "province": ["Brussels", "Liège"],
    "availability_clean": ["Immediately", "Unknown"],
})

# Apply the same transformation to the new data — no re-fitting
new_encoded_array = encoder.transform(new_listings[categorical_cols])

new_encoded_df = pd.DataFrame(
    new_encoded_array,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=new_listings.index
)

# 5. Scale numeric columns - fit on train only, transform both
numeric_cols = ["livable_surface", "number_of_bedrooms", "number_of_bathrooms",
                 "land_surface", "date_of_construction", "latitude", "longitude"]

# Generate a scaler model 
scaler = StandardScaler()
X_train[numeric_cols]=scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]=scaler.transform(X_test[numeric_cols])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print(X_train.dtypes.value_counts())  # sanity check — should be all numeric/bool now

X_train shape: (7284, 49)
X_test shape: (1822, 49)
float64    45
object      4
Name: count, dtype: int64


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

binary_cols = ["elevator", "furnished", "balcony", "swimming_pool"]
categorical_cols = ["property_type", "property_subtype", "property_condition",
                     "province", "availability_clean"]
numeric_cols = ["livable_surface", "number_of_bedrooms", "number_of_bathrooms",
                 "land_surface", "date_of_construction", "latitude", "longitude"]

sale_df_clean[binary_cols] = sale_df_clean[binary_cols].astype(int)

X = sale_df_clean.drop(columns=["price"])
y = sale_df_clean["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoder.fit(X_train[categorical_cols])

def encode(df):
    encoded_array = encoder.transform(df[categorical_cols])
    encoded_df = pd.DataFrame(encoded_array,
        columns=encoder.get_feature_names_out(categorical_cols), index=df.index)
    return pd.concat([df.drop(columns=categorical_cols), encoded_df], axis=1)

X_train_encoded = encode(X_train)
X_test_encoded = encode(X_test)

scaler = StandardScaler()
scaler.fit(X_train_encoded[numeric_cols])

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()
X_train_scaled[numeric_cols] = scaler.transform(X_train_encoded[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test_encoded[numeric_cols])

In [15]:
print(scaler.mean_)   # should match X_train[numeric_cols].mean() before scaling
print(scaler.scale_)  # should match X_train[numeric_cols].std() before scaling

[1.68082372e+02 3.04434377e+00 1.35557386e+00 1.01474973e+03
 1.97254105e+03 5.07169901e+01 4.58522996e+00]
[1.12290936e+02 1.43544394e+00 8.29926896e-01 3.16926269e+03
 3.79527741e+01 3.78509768e-01 7.64725009e-01]


In [16]:
print(X_test[numeric_cols].mean().round(2))
print(X_test[numeric_cols].std().round(2))

livable_surface          174.57
number_of_bedrooms         3.11
number_of_bathrooms        1.37
land_surface            1030.87
date_of_construction    1972.07
latitude                  50.72
longitude                  4.59
dtype: float64
livable_surface          134.37
number_of_bedrooms         1.80
number_of_bathrooms        1.09
land_surface            2674.96
date_of_construction      37.44
latitude                   0.37
longitude                  0.76
dtype: float64


In [17]:
# Verify that the training set has been standardized correctly
print("Trained mean (should be ~0):\n", X_train[numeric_cols].mean())
print("\nTrain std (should be ~1):\n", X_train_scaled[numeric_cols].std(ddof=0))


Trained mean (should be ~0):
 livable_surface          168.082372
number_of_bedrooms         3.044344
number_of_bathrooms        1.355574
land_surface            1014.749725
date_of_construction    1972.541049
latitude                  50.716990
longitude                  4.585230
dtype: float64

Train std (should be ~1):
 livable_surface         1.0
number_of_bedrooms      1.0
number_of_bathrooms     1.0
land_surface            1.0
date_of_construction    1.0
latitude                1.0
longitude               1.0
dtype: float64


binary_cols = ["elevator", "furnished", "balcony", "swimming_pool"]
categorical_cols = ["property_type", "property_subtype", "property_condition",
                     "province", "availability_clean"]
numeric_cols = ["livable_surface", "number_of_bedrooms", "number_of_bathrooms",
                 "land_surface", "date_of_construction", "latitude", "longitude"]



In [18]:
# Better practice : pipline for processing

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify column groups
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

# Build a preprocessor that applies the right transform to each group
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

# Wrap it in a Pipeline (so it's one object — easy to reuse, tune, or extend with a model step)
preprocessing_pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

# Fit on training data only, transform both
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)# YOUR CODE HERE


X_train_processed shape: (7284, 49)
X_test_processed shape: (1822, 49)


/var/folders/rz/ydf5c9rs7mz4fwy7q8pyn89w0000gn/T/ipykernel_3043/794633914.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns.tolist()


In [19]:
print("Numeric:", len(numeric_features))
print("Categorical (raw):", len(categorical_features))
print("Categorical (after one-hot):", X_train_processed.shape[1] - len(numeric_features))

Numeric: 11
Categorical (raw): 5
Categorical (after one-hot): 38


In [20]:
X_train[categorical_features].nunique().sort_values(ascending=False)

property_subtype      15
province              11
property_condition    10
availability_clean     5
property_type          2
dtype: int64

In [21]:
sale_df_clean["property_subtype"] = sale_df_clean["property_subtype"].mask(
    sale_df_clean["property_subtype"].map(sale_df_clean["property_subtype"].value_counts()) < 30,
    "Other"
)

In [22]:
# 1. Bucket rare categories first
sale_df_clean["property_subtype"] = sale_df_clean["property_subtype"].mask(
    sale_df_clean["property_subtype"].map(sale_df_clean["property_subtype"].value_counts()) < 30,
    "Other"
)

# 2. THEN split
X = sale_df_clean.drop(columns=["price"])
y = sale_df_clean["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
X_train[categorical_features].nunique().sort_values(ascending=False)

property_subtype      13
province              11
property_condition    10
availability_clean     5
property_type          2
dtype: int64

In [24]:
from sklearn.linear_model import LinearRegression

# 5. Train a model
def train_model(X_train, y_train):
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

model = train_model(X_train_scaled, y_train)

In [25]:
# 6. Predict
def predict(model, X):
    return model.predict(X)

y_pred_train = predict(model, X_train_scaled)
y_pred_test = predict(model, X_test_scaled)

print("Train predictions shape:", y_pred_train.shape)
print("Test predictions shape:", y_pred_test.shape)

Train predictions shape: (7284,)
Test predictions shape: (1822,)


## Metrics!

In [26]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
import numpy as np

r2 = r2_score(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)

n = len(y_test)
p = X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

mape = mean_absolute_percentage_error(y_test, y_pred_test) * 100


print(f"R-Squared (R²): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Adjusted R-Squared: {adj_r2:.4f}")
print(f"\nWith {p} predictor(s) and {n} samples")
print(f"Difference from R²: {r2 - adj_r2:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

R-Squared (R²): 0.6227
Mean Absolute Error (MAE): 116498.8640
Mean Squared Error (MSE): 50021463438.6398
Root Mean Squared Error (RMSE): 223654.7863
Adjusted R-Squared: 0.6193

With 16 predictor(s) and 1822 samples
Difference from R²: 0.0033
Mean Absolute Percentage Error (MAPE): 32.95%


In [27]:
score = model.score(X_train_scaled, y_train)
print("Evaluating the model on the training set yields an accuracy of {}%".format(score*100))
score=model.score(X_test_scaled, y_test)
print("Evaluating the model on the testing set yields an accuracy of {:.2f}%".format(score*100))

Evaluating the model on the training set yields an accuracy of 64.88321289579835%
Evaluating the model on the testing set yields an accuracy of 62.27%


# No, there is No overfitting!

----------


In [28]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import numpy as np

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mape = mean_absolute_percentage_error(y_test, y_pred_test) * 100
    gap = train_r2 - test_r2
    print(f"--- {name} ---")
    print(f"Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f} (gap: {gap:.4f})")
    print(f"MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}%")
    print("Overfitting:", "Yes, gap is large" if gap > 0.1 else "No significant overfitting")
    return {"model": name, "train_r2": train_r2, "test_r2": test_r2, "mae": mae, "rmse": rmse, "mape": mape}

results = []

In [29]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2,
                                  random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
results.append(evaluate_model("Random Forest", rf_model, X_train_scaled, y_train, X_test_scaled, y_test))

--- Random Forest ---
Train R²: 0.9353 | Test R²: 0.7132 (gap: 0.2220)
MAE: 90,858 | RMSE: 194,980 | MAPE: 24.87%
Overfitting: Yes, gap is large


In [30]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train)
results.append(evaluate_model("XGBoost", xgb_model, X_train_scaled, y_train, X_test_scaled, y_test))

--- XGBoost ---
Train R²: 0.9711 | Test R²: 0.7690 (gap: 0.2020)
MAE: 82,609 | RMSE: 174,988 | MAPE: 22.55%
Overfitting: Yes, gap is large


In [31]:
results_df = pd.DataFrame(results).sort_values("test_r2", ascending=False)
results_df

,model,train_r2,test_r2,mae,rmse,mape
1,XGBoost,0.971079,0.769031,82609.210938,174987.856219,22.548547
0,Random Forest,0.935280,0.713240,90857.546326,194980.245815,24.866770


In [32]:
sale_df_clean.dtypes

longitude               float64
latitude                float64
price                     int64
property_type               str
property_subtype            str
date_of_construction    float64
property_condition          str
livable_surface         float64
number_of_bedrooms      float64
number_of_bathrooms     float64
elevator                  int64
furnished                 int64
province                    str
land_surface            float64
balcony                   int64
swimming_pool             int64
availability_clean          str
dtype: object

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#3. split features/Target
X = sale_df_clean.drop(columns=["price"])
y = sale_df_clean["price"]

# 4. Train/test split Before scaling-avoid leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 1. convert boolean flags to clean 0/1
binary_cols = ["elevator", "furnished", "balcony", "swimming_pool"]
sale_df_clean[binary_cols]= sale_df_clean[binary_cols].astype(int)

#2. One-hot encode categoricals
categorical_cols = ["property_type", "property_subtype", "property_condition",
                     "province", "availability_clean"]

# Fit the encoder on TRAIN categorical columns only
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded_array_train = encoder.fit_transform(X_train[categorical_cols])
encoded_array_test = encoder.transform(X_test[categorical_cols])          # <-- ADDED: encode test too

# Turn the arrays into labeled DataFrames, aligned to the correct index
features_train = pd.DataFrame(encoded_array_train, columns=encoder.get_feature_names_out(categorical_cols),
                               index=X_train.index)                       # <-- FIXED: X_train.index, not sale_df_clean.index
features_test = pd.DataFrame(encoded_array_test, columns=encoder.get_feature_names_out(categorical_cols),
                              index=X_test.index)                         # <-- ADDED

# Drop the raw string columns and attach the one-hot columns
X_train = pd.concat([X_train.drop(columns=categorical_cols), features_train], axis=1)   # <-- ADDED
X_test = pd.concat([X_test.drop(columns=categorical_cols), features_test], axis=1)      # <-- ADDED

# Simulate new, unseen data arriving — the encoder handles it consistently
new_listings = pd.DataFrame({
    "property_type": ["Appartment", "House"],
    "property_subtype": ["Flat", "Villa"],
    "property_condition": ["Excellent", "To renovate"],
    "province": ["Brussels", "Liège"],
    "availability_clean": ["Immediately", "Unknown"],
})

# Apply the same transformation to the new data — no re-fitting
new_encoded_array = encoder.transform(new_listings[categorical_cols])

new_encoded_df = pd.DataFrame(
    new_encoded_array,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=new_listings.index
)

# 5. Scale numeric columns - fit on train only, transform both
numeric_cols = ["livable_surface", "number_of_bedrooms", "number_of_bathrooms",
                 "land_surface", "date_of_construction", "latitude", "longitude"]

# Generate a scaler model 
scaler = StandardScaler()
X_train[numeric_cols]=scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]=scaler.transform(X_test[numeric_cols])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print(X_train.dtypes.value_counts())  # sanity check — should be all numeric/bool now

X_train shape: (7284, 47)
X_test shape: (1822, 47)
float64    43
int64       4
Name: count, dtype: int64


In [37]:
sale_df.dtypes

longitude               float64
latitude                float64
transaction_type            str
price                     int64
property_type               str
property_subtype            str
seller_id                 int64
postal_code               int64
date_of_construction    float64
property_condition          str
livable_surface         float64
number_of_bedrooms      float64
number_of_bathrooms     float64
elevator                 object
terrace                 float64
furnished                object
availability                str
province                    str
street                      str
street_number           float64
garage                  float64
land_surface            float64
energy_consumption      float64
garden                  float64
balcony                  object
swimming_pool            object
dtype: object